# Inspect the TING snapshot

Uses the existing local snapshot; makes no API requests. All summaries concern polygon records, not device coverage or validated outage duration. Python standard library only; select a Python Jupyter kernel in VS Code.

In [1]:
import json
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent
SNAPSHOT = ROOT / 'data' / 'ting' / 'initial'
data = json.loads((SNAPSHOT / 'features.esri.json').read_text())
features = data['features']
rows = [f['attributes'] for f in features]
print(f'{len(rows):,} polygon records')
print('Geometry:', data['geometryType'], 'CRS:', data['spatialReference'])

10,095 polygon records
Geometry: esriGeometryPolygon CRS: {'wkid': 4326}


## Schema and an example

Dates in the source are Unix epoch milliseconds. Geometry uses Esri rings in EPSG:4326 (longitude, latitude), not GeoJSON coordinates. Rings may encode exterior boundaries or holes; do not assume every ring is an independent polygon.

In [2]:
for field in data['fields']:
    print(f"{field['name']:22} {field['type']}")
print(json.dumps(features[0], indent=2))

OBJECTID               esriFieldTypeOID
eventId                esriFieldTypeString
start                  esriFieldTypeDate
modified               esriFieldTypeDate
powerState             esriFieldTypeString
recoveredPercentage    esriFieldTypeDouble
displaydate            esriFieldTypeDate
Shape__Area            esriFieldTypeDouble
Shape__Length          esriFieldTypeDouble
State                  esriFieldTypeString
County                 esriFieldTypeString
Zip                    esriFieldTypeString
{
  "attributes": {
    "OBJECTID": 1,
    "eventId": "ec459122-bd68-4650-8a9d-05266c1d21a4",
    "start": 1743154102349,
    "modified": 1743154113414,
    "powerState": "Outage",
    "recoveredPercentage": 0,
    "displaydate": 1743154200000,
    "Shape__Area": 4.847152149523026e-05,
    "Shape__Length": 0.029961955521603426,
    "State": null,
    "County": null,
    "Zip": null
  },
  "geometry": {
    "rings": [
      [
        [
          -89.9864480602169,
          35.106571677684

## Coverage and quality profile

In [3]:
profile = json.loads((SNAPSHOT / 'profile.json').read_text())
print(json.dumps(profile, indent=2))

{
  "records": 10095,
  "distinct_nonnull_event_ids": 268,
  "event_ids_with_multiple_records": 227,
  "null_counts": {
    "OBJECTID": 0,
    "eventId": 0,
    "start": 0,
    "modified": 0,
    "powerState": 0,
    "recoveredPercentage": 0,
    "displaydate": 0,
    "Shape__Area": 0,
    "Shape__Length": 0,
    "State": 10095,
    "County": 10095,
    "Zip": 10095
  },
  "date_ranges": {
    "start": {
      "min_utc": "2025-03-28T09:28:22.349000+00:00",
      "max_utc": "2025-04-17T15:10:59.698000+00:00"
    },
    "modified": {
      "min_utc": "2025-03-28T09:28:33.414000+00:00",
      "max_utc": "2025-04-17T15:37:01.749000+00:00"
    },
    "displaydate": {
      "min_utc": "2025-03-28T09:30:00+00:00",
      "max_utc": "2025-04-17T19:00:00+00:00"
    }
  },
  "power_states": {
    "Outage": 10095
  },
  "county_labels": {
    "None": 10095
  },
  "missing_geometry": 0,
  "modified_before_start": 0,
  "caveats": [
    "Polygon records are not devices or necessarily distinct outages

## Repeated event IDs

Multiple rows can refer to one event. Inspect spatial parts and timestamps before any deduplication.

In [4]:
counts = Counter(row['eventId'] for row in rows)
print('Distinct event IDs:', len(counts))
print('Most frequent event IDs:', counts.most_common(10))
EVENT_ID = counts.most_common(1)[0][0]  # Replace to inspect another event.
history = sorted((r for r in rows if r['eventId'] == EVENT_ID), key=lambda r: (r['displaydate'], r['OBJECTID']))
for row in history[:20]:
    item = dict(row)
    for field in ('start', 'modified', 'displaydate'):
        item[field] = datetime.fromtimestamp(item[field] / 1000, timezone.utc).isoformat() if item[field] is not None else None
    print(item)

Distinct event IDs: 268
Most frequent event IDs: [('710d0e9f-35aa-4d20-88d1-baa592399668', 1089), ('58c0f930-42d0-48f0-8af7-b5fd22710511', 790), ('92b61195-85eb-4ec4-85bf-5e8ff51fd8b2', 666), ('4686853f-3391-474a-a9f1-1c595b8c02f9', 533), ('d7157003-260a-4d55-a24a-98c8d662074b', 446), ('f4421ebd-1d98-429a-8c5a-8d43e18cbb6c', 372), ('932aea3d-349f-46b0-9d95-dcd2d2f7c25c', 354), ('625cc653-a814-4921-aae9-a8e7cf4355e4', 322), ('58c3aed1-a27c-4d50-ad20-625b589d78b3', 293), ('cb076ccf-1243-4455-8ec4-99863840c507', 280)]
{'OBJECTID': 2507, 'eventId': '710d0e9f-35aa-4d20-88d1-baa592399668', 'start': '2025-04-03T06:19:41.371000+00:00', 'modified': '2025-04-03T06:26:36.299000+00:00', 'powerState': 'Outage', 'recoveredPercentage': 0.1666666716337204, 'displaydate': '2025-04-03T06:30:00+00:00', 'Shape__Area': 0.0013737065382883884, 'Shape__Length': 0.17029745573173188, 'State': None, 'County': None, 'Zip': None}
{'OBJECTID': 2508, 'eventId': '710d0e9f-35aa-4d20-88d1-baa592399668', 'start': '2025-

## Daily record inventory (UTC)

These are counts of records and distinct event IDs by display date, not daily outage prevalence. Dates absent from this inventory have no records; this does not demonstrate restored power.

In [5]:
daily = {}
for row in rows:
    day = datetime.fromtimestamp(row['displaydate'] / 1000, timezone.utc).date().isoformat()
    group = daily.setdefault(day, {'records': 0, 'events': set()})
    group['records'] += 1
    group['events'].add(row['eventId'])
print('UTC date    records  distinct event IDs')
for day, group in sorted(daily.items()):
    print(day, f"{group['records']:8}", f"{len(group['events']):8}")

UTC date    records  distinct event IDs
2025-03-28       73        3
2025-03-29      295       26
2025-03-30      525        9
2025-03-31      395       15
2025-04-01      109        3
2025-04-02      261       34
2025-04-03     3057      106
2025-04-04     1072       21
2025-04-05     1026       36
2025-04-06     1543       32
2025-04-07      678       12
2025-04-08      413        5
2025-04-09      101        2
2025-04-10      124        8
2025-04-11      158        4
2025-04-12      127        5
2025-04-13       60        8
2025-04-15        5        2
2025-04-17       73        3


## Limits and next steps

- All County/State/Zip labels are null: Dyer County requires spatial selection against a boundary.
- All records report Outage. Absence of records is not evidence of restoration.
- Confirm modified/displaydate and recovery-percentage semantics before estimating durations.
- No monitored-device denominator or device locations are exposed.
- Map and validate geometry with a GIS-aware reader before grid aggregation.
- Original attributes and additional ISO UTC date columns are also available in attributes.csv.

In [13]:
import pandas as pd

df = pd.read_csv(SNAPSHOT / "attributes.csv")

In [14]:
df

,OBJECTID,eventId,start,modified,powerState,recoveredPercentage,displaydate,Shape__Area,Shape__Length,State,County,Zip,start_utc,modified_utc,displaydate_utc
0,1,ec459122-bd68-4650-8a9d-05266c1d21a4,1743154102349,1743154113414,Outage,0.0000,1743154200000,0.000048,0.029962,NaN,NaN,NaN,2025-03-28T09:28:22.349000+00:00,2025-03-28T09:28:33.414000+00:00,2025-03-28T09:30:00+00:00
1,2,ec459122-bd68-4650-8a9d-05266c1d21a4,1743154102349,1743154113414,Outage,0.0000,1743154200000,0.000216,0.074283,NaN,NaN,NaN,2025-03-28T09:28:22.349000+00:00,2025-03-28T09:28:33.414000+00:00,2025-03-28T09:30:00+00:00
2,3,ec459122-bd68-4650-8a9d-05266c1d21a4,1743154102349,1743154113414,Outage,0.0000,1743154200000,0.000074,0.035472,NaN,NaN,NaN,2025-03-28T09:28:22.349000+00:00,2025-03-28T09:28:33.414000+00:00,2025-03-28T09:30:00+00:00
3,4,ec459122-bd68-4650-8a9d-05266c1d21a4,1743154102349,1743154738126,Outage,0.4375,1743155100000,0.000048,0.029962,NaN,NaN,NaN,2025-03-28T09:28:22.349000+00:00,2025-03-28T09:38:58.126000+00:00,2025-03-28T09:45:00+00:00
4,5,ec459122-bd68-4650-8a9d-05266c1d21a4,1743154102349,1743154738126,Outage,0.4375,1743155100000,0.000169,0.057329,NaN,NaN,NaN,2025-03-28T09:28:22.349000+00:00,2025-03-28T09:38:58.126000+00:00,2025-03-28T09:45:00+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10090,10091,a3fba79a-89be-46c2-a2bc-014d3427d3d4,1744859347626,1744904221749,Outage,0.6000,1744912800000,0.000014,0.017087,NaN,NaN,NaN,2025-04-17T03:09:07.626000+00:00,2025-04-17T15:37:01.749000+00:00,2025-04-17T18:00:00+00:00
10091,10092,a3fba79a-89be-46c2-a2bc-014d3427d3d4,1744859347626,1744904221749,Outage,0.6000,1744913700000,0.000014,0.017087,NaN,NaN,NaN,2025-04-17T03:09:07.626000+00:00,2025-04-17T15:37:01.749000+00:00,2025-04-17T18:15:00+00:00
10092,10093,a3fba79a-89be-46c2-a2bc-014d3427d3d4,1744859347626,1744904221749,Outage,0.6000,1744914600000,0.000014,0.017087,NaN,NaN,NaN,2025-04-17T03:09:07.626000+00:00,2025-04-17T15:37:01.749000+00:00,2025-04-17T18:30:00+00:00
10093,10094,a3fba79a-89be-46c2-a2bc-014d3427d3d4,1744859347626,1744904221749,Outage,0.6000,1744915500000,0.000014,0.017087,NaN,NaN,NaN,2025-04-17T03:09:07.626000+00:00,2025-04-17T15:37:01.749000+00:00,2025-04-17T18:45:00+00:00
